In [5]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

file_path = "../data/raw/financial_fraud_detection_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Dataset loaded successfully.
Rows    : 5,000
Columns : 14


In [2]:
print("Column Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Column Names:
['Transaction_ID', 'Customer_ID', 'Transaction_Date', 'Transaction_Amount', 'Merchant_Category', 'Payment_Method', 'Device_Type', 'Location', 'Is_International', 'Previous_Transactions', 'Average_Spend', 'Account_Age_Days', 'Suspicious_Keyword', 'Fraudulent']

Data Types:
Transaction_ID               str
Customer_ID                  str
Transaction_Date             str
Transaction_Amount       float64
Merchant_Category            str
Payment_Method               str
Device_Type                  str
Location                     str
Is_International           int64
Previous_Transactions      int64
Average_Spend            float64
Account_Age_Days           int64
Suspicious_Keyword           str
Fraudulent                 int64
dtype: object

Missing Values:
Transaction_ID           0
Customer_ID              0
Transaction_Date         0
Transaction_Amount       0
Merchant_Category        0
Payment_Method           0
Device_Type              0
Location                 0
Is_I

In [6]:
print(df.head())

  Transaction_ID Customer_ID  Transaction_Date  Transaction_Amount  \
0        T100000    CUST3252  04-10-2023 07:45               37.54   
1        T100001    CUST1630  26-09-2023 08:29              240.81   
2        T100002    CUST7852  18-07-2023 15:54              105.34   
3        T100003    CUST4892  16-10-2023 17:10               73.04   
4        T100004    CUST8831  09-09-2023 00:46               13.57   

  Merchant_Category Payment_Method Device_Type   Location  Is_International  \
0            Travel         PayPal         POS  Bengaluru                 0   
1         Utilities         PayPal      Mobile  Bengaluru                 0   
2     Entertainment     Debit Card      Mobile  Bengaluru                 0   
3         Utilities     NetBanking     Desktop    Kolkata                 0   
4         Utilities    Credit Card     Desktop  Bengaluru                 0   

   Previous_Transactions  Average_Spend  Account_Age_Days Suspicious_Keyword  \
0                     94

In [13]:
df["Transaction_Date"] = pd.to_datetime(
    df["Transaction_Date"], 
    format="%d-%m-%y %H-%M")

print(df["Transaction_Date"].dtype)
print(df["Transaction_Date"].min())
print(df["Transaction_Date"].max())

datetime64[us]
2023-01-01 02:16:00
2024-02-21 15:34:00


In [14]:
print("Fraudulent value Counts:")
print(df["Fraudulent"].value_counts())

print("\n Fraudulent Percentage")
print(df["Fraudulent"].value_counts(normalize=True).mul(100).round(2))

Fraudulent value Counts:
Fraudulent
0    4518
1     482
Name: count, dtype: int64

 Fraudulent Percentage
Fraudulent
0    90.36
1     9.64
Name: proportion, dtype: float64


In [15]:
categorical_cols = [
    "Merchant_Category",
    "Payment_Method",
    "Device_Type",
    "Location",
    "Is_International",
    "Suspicious_Keyword"
]

for col in categorical_cols:
    print(f"\n{'=' * 50}")
    print(f"{col}")
    print(df[col].value_counts())


Merchant_Category
Merchant_Category
Food             684
Electronics      650
Entertainment    630
Grocery          629
Fashion          608
Health           606
Travel           599
Utilities        594
Name: count, dtype: int64

Payment_Method
Payment_Method
NetBanking     1037
PayPal         1022
Debit Card     1008
Credit Card     972
UPI             961
Name: count, dtype: int64

Device_Type
Device_Type
Desktop    1679
POS        1673
Mobile     1648
Name: count, dtype: int64

Location
Location
Delhi        758
Mumbai       741
Hyderabad    725
Pune         722
Kolkata      713
Bengaluru    686
Chennai      655
Name: count, dtype: int64

Is_International
Is_International
0    4559
1     441
Name: count, dtype: int64

Suspicious_Keyword
Suspicious_Keyword
No     4740
Yes     260
Name: count, dtype: int64


In [16]:
categorical_cols = [
    "Merchant_Category",
    "Payment_Method",
    "Device_Type",
    "Location",
    "Is_International",
    "Suspicious_Keyword"
]

for col in categorical_cols:
    fraud_rate = (
        df.groupby(col)["Fraudulent"]
        .mean()
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
    )

    print(f"\n{'=' * 50}")
    print(f"Fraud Rate by {col} (%)")
    print(fraud_rate)


Fraud Rate by Merchant_Category (%)
Merchant_Category
Entertainment    11.43
Food             10.82
Fashion          10.20
Health           10.07
Travel            9.35
Grocery           9.22
Utilities         8.42
Electronics       7.54
Name: Fraudulent, dtype: float64

Fraud Rate by Payment_Method (%)
Payment_Method
UPI            10.41
Debit Card     10.22
Credit Card     9.88
PayPal          9.59
NetBanking      8.20
Name: Fraudulent, dtype: float64

Fraud Rate by Device_Type (%)
Device_Type
Desktop    10.24
POS         9.50
Mobile      9.16
Name: Fraudulent, dtype: float64

Fraud Rate by Location (%)
Location
Chennai      10.53
Mumbai       10.53
Bengaluru    10.35
Kolkata      10.24
Hyderabad     8.83
Pune          8.59
Delhi         8.58
Name: Fraudulent, dtype: float64

Fraud Rate by Is_International (%)
Is_International
1    36.96
0     7.00
Name: Fraudulent, dtype: float64

Fraud Rate by Suspicious_Keyword (%)
Suspicious_Keyword
Yes    47.31
No      7.57
Name: Fraudulent, dt

In [17]:
fraud_by_international = (
    df.groupby("Is_International")["Fraudulent"]
    .agg(["count", "sum", "mean"])
)

fraud_by_international["fraud_rate_%"] = (
    fraud_by_international["mean"] * 100
).round(2)

fraud_by_international = fraud_by_international.drop(columns="mean")

print(fraud_by_international)

                  count  sum  fraud_rate_%
Is_International                          
0                  4559  319          7.00
1                   441  163         36.96


In [18]:
numeric_cols = [
    "Transaction_Amount",
    "Previous_Transactions",
    "Average_Spend",
    "Account_Age_Days"
]

print(
    df.groupby("Fraudulent")[numeric_cols]
      .agg(["mean", "median", "min", "max"])
      .round(2)
)

           Transaction_Amount                     Previous_Transactions  \
                         mean median  min     max                  mean   
Fraudulent                                                                
0                       78.88  55.46  0.0  595.34                 99.69   
1                       81.99  55.43  0.0  653.80                 96.86   

                           Average_Spend                         \
           median min  max          mean  median    min     max   
Fraudulent                                                        
0           100.0   1  199        258.33  261.64  10.01  499.99   
1            96.0   1  199        258.86  264.12  10.42  499.64   

           Account_Age_Days                    
                       mean  median min   max  
Fraudulent                                     
0                   1009.57  1009.0  30  1999  
1                   1026.06  1054.0  38  1996  


In [19]:
df["Transaction_Hour"] = df["Transaction_Date"].dt.hour

hourly_fraud = (
    df.groupby("Transaction_Hour")["Fraudulent"]
      .agg(["count", "sum", "mean"])
)

hourly_fraud["fraud_rate_%"] = (
    hourly_fraud["mean"] * 100
).round(2)

hourly_fraud = hourly_fraud.drop(columns="mean")

print(hourly_fraud)

                  count  sum  fraud_rate_%
Transaction_Hour                          
0                   188   57         30.32
1                   204   48         23.53
2                   216   44         20.37
3                   200   54         27.00
4                   178   46         25.84
5                   236   64         27.12
6                   225   17          7.56
7                   208    7          3.37
8                   209   10          4.78
9                   220   11          5.00
10                  202    6          2.97
11                  205   10          4.88
12                  211   11          5.21
13                  233   13          5.58
14                  206    7          3.40
15                  196    8          4.08
16                  199   10          5.03
17                  219    5          2.28
18                  212    8          3.77
19                  198   10          5.05
20                  184    6          3.26
21         

In [20]:
interaction_analysis = (
    df.groupby(["Is_International", "Suspicious_Keyword"])["Fraudulent"]
      .agg(["count", "sum", "mean"])
)

interaction_analysis["fraud_rate_%"] = (
    interaction_analysis["mean"] * 100
).round(2)

interaction_analysis = interaction_analysis.drop(columns="mean")

print(interaction_analysis)

                                     count  sum  fraud_rate_%
Is_International Suspicious_Keyword                          
0                No                   4321  215          4.98
                 Yes                   238  104         43.70
1                No                    419  144         34.37
                 Yes                    22   19         86.36


In [21]:
customer_analysis = (
    df.groupby("Customer_ID")["Fraudulent"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

customer_analysis["fraud_rate_%"] = (
    customer_analysis["fraud_rate"] * 100
).round(2)

customer_analysis = customer_analysis.drop(columns="fraud_rate")

print("Number of unique customers:", customer_analysis.shape[0])

print("\nCustomers with the most transactions:")
print(
    customer_analysis
    .sort_values("transaction_count", ascending=False)
    .head(10)
)

print("\nCustomers with the highest fraud counts:")
print(
    customer_analysis
    .sort_values("fraud_count", ascending=False)
    .head(10)
)

Number of unique customers: 3847

Customers with the most transactions:
             transaction_count  fraud_count  fraud_rate_%
Customer_ID                                              
CUST8944                     5            1          20.0
CUST5258                     4            0           0.0
CUST3443                     4            0           0.0
CUST3647                     4            0           0.0
CUST8205                     4            0           0.0
CUST9174                     4            0           0.0
CUST4691                     4            0           0.0
CUST8574                     4            0           0.0
CUST5942                     4            0           0.0
CUST2246                     4            1          25.0

Customers with the highest fraud counts:
             transaction_count  fraud_count  fraud_rate_%
Customer_ID                                              
CUST8610                     3            2         66.67
CUST7972        

In [22]:
for col in ["Device_Type", "Location"]:
    result = (
        df.groupby(col)["Fraudulent"]
          .agg(["count", "sum", "mean"])
    )

    result["fraud_rate_%"] = (
        result["mean"] * 100
    ).round(2)

    result = result.drop(columns="mean")

    print(f"\n{'=' * 50}")
    print(f"Fraud by {col}")
    print(result.sort_values("fraud_rate_%", ascending=False))


Fraud by Device_Type
             count  sum  fraud_rate_%
Device_Type                          
Desktop       1679  172         10.24
POS           1673  159          9.50
Mobile        1648  151          9.16

Fraud by Location
           count  sum  fraud_rate_%
Location                           
Chennai      655   69         10.53
Mumbai       741   78         10.53
Bengaluru    686   71         10.35
Kolkata      713   73         10.24
Hyderabad    725   64          8.83
Pune         722   62          8.59
Delhi        758   65          8.58


In [23]:
numeric_cols = [
    "Transaction_Amount",
    "Previous_Transactions",
    "Average_Spend",
    "Account_Age_Days"
]

print(
    df.groupby("Fraudulent")[numeric_cols]
      .describe()
      .round(2)
)

           Transaction_Amount                                           \
                        count   mean    std  min    25%    50%     75%   
Fraudulent                                                               
0                      4518.0  78.88  77.78  0.0  22.64  55.46  110.37   
1                       482.0  81.99  89.37  0.0  21.49  55.43  109.29   

                   Previous_Transactions                                  \
               max                 count   mean    std  min   25%    50%   
Fraudulent                                                                 
0           595.34                4518.0  99.69  56.96  1.0  50.0  100.0   
1           653.80                 482.0  96.86  57.59  1.0  44.0   96.0   

                         Average_Spend                                         \
              75%    max         count    mean     std    min     25%     50%   
Fraudulent                                                                      
0    

In [24]:
print(df["Transaction_Amount"].describe())

print("\nPotential extreme transactions:")
print(
    df.nlargest(10, "Transaction_Amount")[
        ["Transaction_ID", "Transaction_Amount", "Fraudulent"]
    ]
)

count    5000.000000
mean       79.180664
std        78.964045
min         0.000000
25%        22.365000
50%        55.455000
75%       110.297500
max       653.800000
Name: Transaction_Amount, dtype: float64

Potential extreme transactions:
     Transaction_ID  Transaction_Amount  Fraudulent
531         T100531              653.80           1
3649        T103649              617.88           1
1464        T101464              595.34           0
3402        T103402              544.46           0
1954        T101954              512.44           1
1054        T101054              494.58           0
2529        T102529              490.31           0
4256        T104256              487.72           1
3407        T103407              483.53           0
3502        T103502              471.34           0


In [25]:
amount_bins = [0, 25, 50, 100, 200, 400, float("inf")]

amount_labels = [
    "0-25",
    "25-50",
    "50-100",
    "100-200",
    "200-400",
    "400+"
]

df["Amount_Band"] = pd.cut(
    df["Transaction_Amount"],
    bins=amount_bins,
    labels=amount_labels,
    include_lowest=True
)

amount_analysis = (
    df.groupby("Amount_Band", observed=False)["Fraudulent"]
      .agg(["count", "sum", "mean"])
)

amount_analysis["fraud_rate_%"] = (
    amount_analysis["mean"] * 100
).round(2)

amount_analysis = amount_analysis.drop(columns="mean")

print(amount_analysis)

             count  sum  fraud_rate_%
Amount_Band                          
0-25          1360  139         10.22
25-50          989   85          8.59
50-100        1243  126         10.14
100-200       1006   90          8.95
200-400        372   36          9.68
400+            30    6         20.00


In [26]:
# Extract the transaction hour
df["Transaction_Hour"] = df["Transaction_Date"].dt.hour

# Calculate fraud statistics by hour
hour_analysis = (
    df.groupby("Transaction_Hour")["Fraudulent"]
      .agg(["count", "sum", "mean"])
)

# Convert fraud rate into percentage
hour_analysis["fraud_rate_%"] = (
    hour_analysis["mean"] * 100
).round(2)

# Remove the intermediate mean column
hour_analysis = hour_analysis.drop(columns="mean")

print(hour_analysis)

                  count  sum  fraud_rate_%
Transaction_Hour                          
0                   188   57         30.32
1                   204   48         23.53
2                   216   44         20.37
3                   200   54         27.00
4                   178   46         25.84
5                   236   64         27.12
6                   225   17          7.56
7                   208    7          3.37
8                   209   10          4.78
9                   220   11          5.00
10                  202    6          2.97
11                  205   10          4.88
12                  211   11          5.21
13                  233   13          5.58
14                  206    7          3.40
15                  196    8          4.08
16                  199   10          5.03
17                  219    5          2.28
18                  212    8          3.77
19                  198   10          5.05
20                  184    6          3.26
21         

In [27]:
# Fraud analysis by international transaction status

international_analysis = (
    df.groupby("Is_International")["Fraudulent"]
      .agg(["count", "sum", "mean"])
)

# Convert fraud rate into percentage
international_analysis["fraud_rate_%"] = (
    international_analysis["mean"] * 100
).round(2)

# Remove the intermediate mean column
international_analysis = international_analysis.drop(columns="mean")

print(international_analysis)

                  count  sum  fraud_rate_%
Is_International                          
0                  4559  319          7.00
1                   441  163         36.96


In [28]:
# Analyze fraud rate by international status and suspicious keyword

interaction_analysis = (
    df.groupby(
        ["Is_International", "Suspicious_Keyword"]
    )["Fraudulent"]
    .agg(["count", "sum", "mean"])
)

# Convert fraud rate into percentage
interaction_analysis["fraud_rate_%"] = (
    interaction_analysis["mean"] * 100
).round(2)

# Remove the intermediate mean column
interaction_analysis = interaction_analysis.drop(columns="mean")

print(interaction_analysis)

                                     count  sum  fraud_rate_%
Is_International Suspicious_Keyword                          
0                No                   4321  215          4.98
                 Yes                   238  104         43.70
1                No                    419  144         34.37
                 Yes                    22   19         86.36


In [29]:
interaction_analysis = interaction_analysis.reset_index()

interaction_analysis["Transaction_Type"] = (
    interaction_analysis["Is_International"]
    .map({
        0: "Domestic",
        1: "International"
    })
)

print(interaction_analysis)

   Is_International Suspicious_Keyword  count  sum  fraud_rate_%  \
0                 0                 No   4321  215          4.98   
1                 0                Yes    238  104         43.70   
2                 1                 No    419  144         34.37   
3                 1                Yes     22   19         86.36   

  Transaction_Type  
0         Domestic  
1         Domestic  
2    International  
3    International  


In [30]:
# Fraud analysis by payment method

payment_analysis = (
    df.groupby("Payment_Method")["Fraudulent"]
      .agg(["count", "sum", "mean"])
)

payment_analysis["fraud_rate_%"] = (
    payment_analysis["mean"] * 100
).round(2)

payment_analysis = payment_analysis.drop(columns="mean")

print(payment_analysis)

                count  sum  fraud_rate_%
Payment_Method                          
Credit Card       972   96          9.88
Debit Card       1008  103         10.22
NetBanking       1037   85          8.20
PayPal           1022   98          9.59
UPI               961  100         10.41


In [31]:
# Fraud analysis by device type

device_analysis = (
    df.groupby("Device_Type")["Fraudulent"]
      .agg(["count", "sum", "mean"])
)

# Convert fraud rate into percentage
device_analysis["fraud_rate_%"] = (
    device_analysis["mean"] * 100
).round(2)

# Remove the intermediate mean column
device_analysis = device_analysis.drop(columns="mean")

print(device_analysis)

             count  sum  fraud_rate_%
Device_Type                          
Desktop       1679  172         10.24
Mobile        1648  151          9.16
POS           1673  159          9.50


In [32]:
# Fraud analysis by merchant category

merchant_analysis = (
    df.groupby("Merchant_Category")["Fraudulent"]
      .agg(["count", "sum", "mean"])
)

# Convert fraud rate into percentage
merchant_analysis["fraud_rate_%"] = (
    merchant_analysis["mean"] * 100
).round(2)

# Remove the intermediate mean column
merchant_analysis = merchant_analysis.drop(columns="mean")

print(merchant_analysis)

                   count  sum  fraud_rate_%
Merchant_Category                          
Electronics          650   49          7.54
Entertainment        630   72         11.43
Fashion              608   62         10.20
Food                 684   74         10.82
Grocery              629   58          9.22
Health               606   61         10.07
Travel               599   56          9.35
Utilities            594   50          8.42


In [33]:
# Group previous transactions into bands

previous_bins = [0, 25, 50, 100, 150, 200]

previous_labels = [
    "1-25",
    "26-50",
    "51-100",
    "101-150",
    "151-200"
]

df["Previous_Transactions_Band"] = pd.cut(
    df["Previous_Transactions"],
    bins=previous_bins,
    labels=previous_labels,
    include_lowest=True
)

previous_analysis = (
    df.groupby(
        "Previous_Transactions_Band",
        observed=False
    )["Fraudulent"]
    .agg(["count", "sum", "mean"])
)

previous_analysis["fraud_rate_%"] = (
    previous_analysis["mean"] * 100
).round(2)

previous_analysis = previous_analysis.drop(columns="mean")

print(previous_analysis)

                            count  sum  fraud_rate_%
Previous_Transactions_Band                          
1-25                          621   73         11.76
26-50                         642   57          8.88
51-100                       1251  119          9.51
101-150                      1295  120          9.27
151-200                      1191  113          9.49


In [34]:
# Group average spend into bands

spend_bins = [0, 50, 100, 200, 300, 400, 500]

spend_labels = [
    "0-50",
    "50-100",
    "100-200",
    "200-300",
    "300-400",
    "400-500"
]

df["Average_Spend_Band"] = pd.cut(
    df["Average_Spend"],
    bins=spend_bins,
    labels=spend_labels,
    include_lowest=True
)

spend_analysis = (
    df.groupby(
        "Average_Spend_Band",
        observed=False
    )["Fraudulent"]
    .agg(["count", "sum", "mean"])
)

spend_analysis["fraud_rate_%"] = (
    spend_analysis["mean"] * 100
).round(2)

spend_analysis = spend_analysis.drop(columns="mean")

print(spend_analysis)

                    count  sum  fraud_rate_%
Average_Spend_Band                          
0-50                  407   52         12.78
50-100                480   37          7.71
100-200              1002   87          8.68
200-300              1007  102         10.13
300-400              1070   98          9.16
400-500              1034  106         10.25


In [35]:
# Check data types
print("Data Types:\n")
print(df.dtypes)

print("\n" + "=" * 60)

# Check missing values
print("\nMissing Values:\n")
print(df.isnull().sum())

print("\n" + "=" * 60)

# Check duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

Data Types:

Transaction_ID                           str
Customer_ID                              str
Transaction_Date              datetime64[us]
Transaction_Amount                   float64
Merchant_Category                        str
Payment_Method                           str
Device_Type                              str
Location                                 str
Is_International                       int64
Previous_Transactions                  int64
Average_Spend                        float64
Account_Age_Days                       int64
Suspicious_Keyword                       str
Fraudulent                             int64
Transaction_Hour                       int32
Amount_Band                         category
Previous_Transactions_Band          category
Average_Spend_Band                  category
dtype: object


Missing Values:

Transaction_ID                0
Customer_ID                   0
Transaction_Date              0
Transaction_Amount            0
Merchant_Categor

In [36]:
# Unique value count for each column
unique_summary = pd.DataFrame({
    "Column": df.columns,
    "Unique_Values": df.nunique().values
})

print(unique_summary)

                        Column  Unique_Values
0               Transaction_ID           5000
1                  Customer_ID           3847
2             Transaction_Date           4980
3           Transaction_Amount           4302
4            Merchant_Category              8
5               Payment_Method              5
6                  Device_Type              3
7                     Location              7
8             Is_International              2
9        Previous_Transactions            199
10               Average_Spend           4753
11            Account_Age_Days           1827
12          Suspicious_Keyword              2
13                  Fraudulent              2
14            Transaction_Hour             24
15                 Amount_Band              6
16  Previous_Transactions_Band              5
17          Average_Spend_Band              6


In [38]:
# Target distribution
target_summary = df["Fraudulent"].value_counts().reset_index()

target_summary.columns = ["Fraudulent", "Count"]

target_summary["Percentage"] = (
    target_summary["Count"] / len(df) * 100
).round(2)

print(target_summary)

   Fraudulent  Count  Percentage
0           0   4518       90.36
1           1    482        9.64


In [39]:
# Convert transaction date
df["Transaction_Date"] = pd.to_datetime(
    df["Transaction_Date"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

print("Invalid Dates:", df["Transaction_Date"].isna().sum())

print("\nMinimum Date:", df["Transaction_Date"].min())
print("Maximum Date:", df["Transaction_Date"].max())

Invalid Dates: 0

Minimum Date: 2023-01-01 02:16:00
Maximum Date: 2024-02-21 15:34:00


In [40]:
# Select numerical columns
numeric_columns = [
    "Transaction_Amount",
    "Is_International",
    "Previous_Transactions",
    "Average_Spend",
    "Account_Age_Days",
    "Fraudulent"
]

# Calculate correlation matrix
correlation_matrix = df[numeric_columns].corr()

print(correlation_matrix.round(3))

                       Transaction_Amount  Is_International  \
Transaction_Amount                  1.000            -0.005   
Is_International                   -0.005             1.000   
Previous_Transactions              -0.011            -0.006   
Average_Spend                       0.025             0.020   
Account_Age_Days                    0.003             0.006   
Fraudulent                          0.012             0.288   

                       Previous_Transactions  Average_Spend  Account_Age_Days  \
Transaction_Amount                    -0.011          0.025             0.003   
Is_International                      -0.006          0.020             0.006   
Previous_Transactions                  1.000         -0.017            -0.001   
Average_Spend                         -0.017          1.000            -0.003   
Account_Age_Days                      -0.001         -0.003             1.000   
Fraudulent                            -0.015          0.001             

In [42]:
# Customer transaction frequency

customer_frequency = (
    df.groupby("Customer_ID")
      .size()
      .reset_index(name="Transaction_Count")
)

print(customer_frequency.head(10))

print("\nFrequency Summary:")
print(customer_frequency["Transaction_Count"].describe())

  Customer_ID  Transaction_Count
0    CUST1000                  1
1    CUST1001                  1
2    CUST1002                  1
3    CUST1004                  2
4    CUST1005                  1
5    CUST1007                  1
6    CUST1009                  1
7    CUST1013                  1
8    CUST1014                  1
9    CUST1015                  3

Frequency Summary:
count    3847.000000
mean        1.299714
std         0.573607
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         5.000000
Name: Transaction_Count, dtype: float64


In [43]:
# Distribution of transactions per customer

frequency_distribution = (
    customer_frequency["Transaction_Count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

frequency_distribution.columns = [
    "Transaction_Count",
    "Customer_Count"
]

print(frequency_distribution)

   Transaction_Count  Customer_Count
0                  1            2895
1                  2             778
2                  3             148
3                  4              25
4                  5               1


In [44]:
# Add customer transaction count to the working data

df_frequency = df.merge(
    customer_frequency,
    on="Customer_ID",
    how="left"
)

frequency_fraud_summary = (
    df_frequency.groupby("Transaction_Count")["Fraudulent"]
    .agg(
        Transaction_Count_Total="count",
        Fraud_Count="sum",
        Fraud_Rate="mean"
    )
    .reset_index()
)

frequency_fraud_summary["Fraud_Rate"] = (
    frequency_fraud_summary["Fraud_Rate"] * 100
).round(2)

print(frequency_fraud_summary)

   Transaction_Count  Transaction_Count_Total  Fraud_Count  Fraud_Rate
0                  1                     2895          284        9.81
1                  2                     1556          154        9.90
2                  3                      444           41        9.23
3                  4                      100            2        2.00
4                  5                        5            1       20.00
